# Auxiliary signals of PGA-UNet on FracAtlas (IMG_SIZE=512)

Two quantitative evaluations of PGA-UNet's no-ground-truth outputs, both on the
held-out test split with the official PGA-512 checkpoint (trained with
`USE_QUALITY_HEAD=1`).

**Part A - QualityHead predictive validity (Claim 9).** Per lesion prompt, compare
the QualityHead score against the true per-polygon Dice. Reports MAE, RMSE, Pearson,
Spearman and a binned reliability curve, separately for `center_zoom` and
`center_shift`.

**Part B - candidate region suggestion (Claim 10).** For each test image, sample 50
candidate boxes, rank them by the QualityHead score, keep the top 5, and measure how
well those boxes geometrically cover the ground-truth lesion pixels. No predicted
mask is used for scoring: this evaluates the suggested *region*, not a mask.
The QualityHead score is an auxiliary estimate, not a calibrated probability, and the
suggestion is a shortlist for clinician review, not automatic lesion detection.

Run in order: **Setup -> Part A -> Part B**.

In [ ]:
# -- Setup ---------------------------------------------------------------
%cd /kaggle/working
import os, gdown, torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Clone repo
if not os.path.exists('PGA_Unet2D'):
    !git clone --branch main --single-branch https://github.com/ThongLuc2k3/PGA_Unet2D.git
else:
    !cd PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation && git pull -q

PGA_PATH = '/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'

# Dataset
DATASET_ID   = '1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv'
DATASET_ROOT = 'dataset_FracAtlas'
if not os.path.exists(f'/kaggle/working/{DATASET_ROOT}'):
    gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}',
                   f'/kaggle/working/{DATASET_ROOT}.zip', quiet=False)
    !unzip -oq /kaggle/working/{DATASET_ROOT}.zip -d /kaggle/working/
!rsync -a /kaggle/working/{DATASET_ROOT}/ {PGA_PATH}/{DATASET_ROOT}/ 2>/dev/null

os.chdir(PGA_PATH)
!pip install -q tqdm opencv-python matplotlib gdown scipy

# -- Checkpoint: official PGA-512 (folder `00`, Dice+BCE, QualityHead) ---
CKPT_ID   = '1ZUyhKEKqCZmPyGDBBOpsfQ7BuqJGFBhU'
CKPT_PATH = f'{PGA_PATH}/checkpoints/pga_unet_center_mixed_x3_shift05_qhead_512_best.pth'
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
assert CKPT_ID, 'Fill CKPT_ID with the PGA-512 `00` checkpoint (must have QualityHead)'
if not os.path.exists(CKPT_PATH):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_PATH, quiet=False)
assert os.path.exists(CKPT_PATH)

_state = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)
assert any(k.startswith('quality_head.') for k in _state), \
    'This checkpoint has no QualityHead weights; retrain with USE_QUALITY_HEAD=1.'
print(f'\nSetup complete | checkpoint {os.path.getsize(CKPT_PATH)//1024} KB, QualityHead present')

## Part A - QualityHead predictive validity (Claim 9)

Per lesion prompt: `quality_pred` (QualityHead) vs `true_dice` (thresholded Dice of
that prompt's predicted mask against its polygon). Image-level merging is not used
here because QualityHead is a per-prompt signal.

In [ ]:
# -- Part A: QualityHead vs true per-polygon Dice -----------------------
import sys, csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset', 'models', 'prompt_unet')):
        del sys.modules[_k]
if PGA_PATH in sys.path: sys.path.remove(PGA_PATH)
sys.path.insert(0, PGA_PATH)
from dataset import PromptSegmentationDataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
DS_NAME  = 'FracAtlas'
TEST_IMG  = f'{PGA_PATH}/{DATASET_ROOT}/test/images'
TEST_JSON = f'{PGA_PATH}/{DATASET_ROOT}/test/annotations'
RESULT_DIR = f'{PGA_PATH}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True, use_quality_head=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()

DICE_THRESHOLD = 0.7

def per_sample_dice(logits, target, smooth=1e-5):
    pred = (torch.sigmoid(logits) > 0.5).float()
    tp = (pred * target).sum(dim=(1, 2, 3))
    fp = (pred * (1 - target)).sum(dim=(1, 2, 3))
    fn = ((1 - pred) * target).sum(dim=(1, 2, 3))
    return (2 * tp + smooth) / (2 * tp + fp + fn + smooth)

def corr(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or a.std() == 0 or b.std() == 0:
        return float('nan'), float('nan')
    pear = float(np.corrcoef(a, b)[0, 1])
    ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))
    spear = float(np.corrcoef(ra, rb)[0, 1])
    return pear, spear

rows = []
summary = []
for prompt_mode in ('center_zoom', 'center_shift'):
    ds = PromptSegmentationDataset(TEST_IMG, TEST_JSON, img_size=IMG_SIZE, is_train=False,
                                  prompt_mode=prompt_mode, scale_factor=3.0, shift_ratio=0.5)
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
    q_all, d_all = [], []
    offset = 0
    with torch.no_grad():
        for images, masks, prompts in loader:
            images, masks, prompts = images.to(DEVICE), masks.to(DEVICE), prompts.to(DEVICE)
            logits, quality = model(images, prompts, return_quality=True)
            dice = per_sample_dice(logits, masks).cpu().numpy()
            quality = quality.detach().cpu().numpy().reshape(-1)
            for j, (qp, dd) in enumerate(zip(quality, dice)):
                img_name, poly_idx = ds.all_samples[offset + j]
                rows.append(dict(dataset=DS_NAME, prompt=prompt_mode, image=img_name,
                                 polygon_index=poly_idx, quality_pred=float(qp),
                                 true_dice=float(dd), abs_error=float(abs(qp - dd)),
                                 usable=int(dd >= DICE_THRESHOLD)))
            q_all.extend(quality.tolist()); d_all.extend(dice.tolist())
            offset += len(dice)
    q_all, d_all = np.array(q_all), np.array(d_all)
    pear, spear = corr(q_all, d_all)
    summary.append(dict(dataset=DS_NAME, prompt=prompt_mode, n=len(q_all),
                        mae=float(np.mean(np.abs(q_all - d_all))),
                        rmse=float(np.sqrt(np.mean((q_all - d_all) ** 2))),
                        pearson=pear, spearman=spear))

print(f'{"dataset":<10}{"prompt":<14}{"N":>5}{"MAE":>9}{"RMSE":>9}{"Pearson":>10}{"Spearman":>10}')
print('-' * 67)
for s in summary:
    print(f'{s["dataset"]:<10}{s["prompt"]:<14}{s["n"]:>5}{s["mae"]:>9.4f}{s["rmse"]:>9.4f}'
          f'{s["pearson"]:>10.4f}{s["spearman"]:>10.4f}')

with open(f'{RESULT_DIR}/qualityhead_per_polygon_{DS_NAME.lower()}.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(f'{RESULT_DIR}/qualityhead_summary_{DS_NAME.lower()}.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys())); w.writeheader(); w.writerows(summary)

# -- scatter + binned reliability (center_shift) --
shift = [r for r in rows if r['prompt'] == 'center_shift']
qp = np.array([r['quality_pred'] for r in shift]); td = np.array([r['true_dice'] for r in shift])
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(qp, td, s=12, alpha=0.5)
ax[0].plot([0, 1], [0, 1], 'k--', lw=1)
ax[0].set_xlabel('QualityHead score'); ax[0].set_ylabel('true Dice')
ax[0].set_title(f'{DS_NAME} center_shift (n={len(shift)})'); ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1)
edges = np.arange(0, 1.01, 0.1); centers = (edges[:-1] + edges[1:]) / 2
mean_dice = [td[(qp >= lo) & (qp < hi)].mean() if ((qp >= lo) & (qp < hi)).any() else np.nan
             for lo, hi in zip(edges[:-1], edges[1:])]
ax[1].plot(centers, mean_dice, 'o-')
ax[1].plot([0, 1], [0, 1], 'k--', lw=1)
ax[1].set_xlabel('QualityHead score bin'); ax[1].set_ylabel('mean true Dice in bin')
ax[1].set_title('Binned reliability'); ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/qualityhead_reliability_{DS_NAME.lower()}.png', dpi=130, facecolor='white')
plt.show()

## Part B - candidate region suggestion (Claim 10)

For each test image: sample `N_CANDIDATES` boxes (side 0.15 to 0.55 of the 512
canvas, uniform position), score each with the QualityHead, keep the top `TOP_K`.

`coverage` of a candidate box = (ground-truth lesion pixels inside the box) / (all
ground-truth lesion pixels in the image). A box that fully wraps the lesion scores
1.0; a box that catches half scores 0.5; no predicted mask is involved.

Per image: `mean_cov` over the top-k boxes, and `best_cov` = the single best box.
Reported per dataset (mean over images):
- **mean coverage** = mean of `mean_cov`
- **mean best coverage** = mean of `best_cov`
- **full-coverage rate** = fraction of images whose `best_cov` reaches 1.0 (at least
  one suggested box wraps the whole lesion)

Shown for top-3 and top-5 from the same ranked list.

In [ ]:
# -- Part B: candidate region suggestion -------------------------------
import json as _json, cv2
import numpy as np
import torch

N_CANDIDATES = 50
TOP_KS       = (3, 5)
SIZE_FRAC    = (0.15, 0.55)
SEED         = 22120196
CANVAS       = 512
KERNEL       = 31

def resize_and_pad(arr, size, interp, pad_value=0):
    h, w = arr.shape[:2]
    scale = min(size / w, size / h)
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    r = cv2.resize(arr, (nw, nh), interpolation=interp)
    out = np.full((size, size), pad_value, dtype=r.dtype)
    pl, pt = (size - nw) // 2, (size - nh) // 2
    out[pt:pt + nh, pl:pl + nw] = r
    return out

def plateau_heatmap(bbox, size, kernel=KERNEL):
    hm = np.zeros((size, size), dtype=np.float32)
    x0, y0, x1, y1 = [int(v) for v in bbox]
    x0, y0 = max(0, x0), max(0, y0); x1, y1 = min(size, x1), min(size, y1)
    if x1 > x0 and y1 > y0:
        hm[y0:y1, x0:x1] = 1.0
        hm = cv2.GaussianBlur(hm, (kernel, kernel), 0)
    return hm

def load_image_and_gt(img_name):
    base = os.path.splitext(img_name)[0]
    img = cv2.imread(os.path.join(TEST_IMG, img_name), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    gt = np.zeros((h, w), dtype=np.uint8)
    with open(os.path.join(TEST_JSON, base + '.json'), encoding='utf-8') as f:
        data = _json.load(f)
    for s in data.get('shapes', []):
        if s.get('shape_type') == 'polygon':
            cv2.fillPoly(gt, [np.array(s['points'], dtype=np.int32)], 1)
    canvas = resize_and_pad(img, CANVAS, cv2.INTER_LINEAR, 0)
    canvas = (canvas.astype(np.float32) / 255.0 - 0.5) / 0.5
    gt512 = resize_and_pad(gt.astype(np.float32), CANVAS, cv2.INTER_NEAREST, 0.0)
    gt512 = (gt512 > 0.5).astype(np.float32)
    return canvas, gt512

def score_candidates(canvas, boxes, batch=16):
    img_t = torch.from_numpy(canvas).unsqueeze(0).unsqueeze(0).to(DEVICE)
    scores = []
    with torch.no_grad():
        for i in range(0, len(boxes), batch):
            chunk = boxes[i:i + batch]
            hm = np.stack([plateau_heatmap(b, CANVAS) for b in chunk])
            hm_t = torch.from_numpy(hm).unsqueeze(1).float().to(DEVICE)
            img_rep = img_t.expand(len(chunk), 1, CANVAS, CANVAS)
            _, quality = model(img_rep, hm_t, return_quality=True)
            scores.extend(quality.detach().cpu().numpy().reshape(-1).tolist())
    return np.array(scores)

def box_coverage(box, gt512):
    total = gt512.sum()
    if total == 0:
        return None
    x0, y0, x1, y1 = [int(round(v)) for v in box]
    x0, y0 = max(0, x0), max(0, y0); x1, y1 = min(CANVAS, x1), min(CANVAS, y1)
    if x1 <= x0 or y1 <= y0:
        return 0.0
    return float(gt512[y0:y1, x0:x1].sum() / total)

rng = np.random.default_rng(SEED)
image_names = sorted(f for f in os.listdir(TEST_IMG)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))
                     and os.path.exists(os.path.join(TEST_JSON, os.path.splitext(f)[0] + '.json')))

per_image = []
suggestion_records = []   # kept for the qualitative figure
for img_name in image_names:
    canvas, gt512 = load_image_and_gt(img_name)
    if gt512.sum() == 0:
        continue
    boxes = []
    for _ in range(N_CANDIDATES):
        bw = rng.uniform(*SIZE_FRAC) * CANVAS
        bh = rng.uniform(*SIZE_FRAC) * CANVAS
        bx = rng.uniform(0, CANVAS - bw)
        by = rng.uniform(0, CANVAS - bh)
        boxes.append([bx, by, bx + bw, by + bh])
    scores = score_candidates(canvas, boxes)
    order = np.argsort(-scores)
    cov_all = np.array([box_coverage(b, gt512) for b in boxes])
    rec = dict(dataset=DS_NAME, image=img_name)
    for k in TOP_KS:
        topk = order[:k]
        cov_k = cov_all[topk]
        rec[f'mean_cov@{k}'] = float(cov_k.mean())
        rec[f'best_cov@{k}'] = float(cov_k.max())
        rec[f'full@{k}'] = int(cov_k.max() >= 0.999)
    per_image.append(rec)
    if len(suggestion_records) < 8:   # keep a few for the qualitative figure only
        suggestion_records.append(dict(img_name=img_name, canvas=canvas, gt512=gt512,
                                       boxes=[boxes[i] for i in order[:5]],
                                       scores=[float(scores[i]) for i in order[:5]],
                                       covs=[float(cov_all[i]) for i in order[:5]]))

import csv
with open(f'{RESULT_DIR}/prompt_suggestion_per_image_{DS_NAME.lower()}.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(per_image[0].keys())); w.writeheader(); w.writerows(per_image)

print(f'{DS_NAME}: {len(per_image)} test images with a lesion\n')
print(f'{"top-k":<7}{"mean coverage":>15}{"mean best coverage":>20}{"full-coverage rate":>20}')
print('-' * 62)
agg_rows = []
for k in TOP_KS:
    mc = np.mean([r[f'mean_cov@{k}'] for r in per_image])
    bc = np.mean([r[f'best_cov@{k}'] for r in per_image])
    fr = np.mean([r[f'full@{k}'] for r in per_image])
    print(f'top-{k:<3}{mc:>15.4f}{bc:>20.4f}{fr:>19.1%}')
    agg_rows.append(dict(dataset=DS_NAME, top_k=k, mean_coverage=float(mc),
                         mean_best_coverage=float(bc), full_coverage_rate=float(fr)))
with open(f'{RESULT_DIR}/prompt_suggestion_summary_{DS_NAME.lower()}.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(agg_rows[0].keys())); w.writeheader(); w.writerows(agg_rows)

In [ ]:
# -- Part B: qualitative figure - top-3 suggested boxes on a few images --
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

picks = suggestion_records[:4]
fig, axes = plt.subplots(1, len(picks), figsize=(4.2 * len(picks), 4.4))
if len(picks) == 1:
    axes = [axes]
for ax, rec in zip(axes, picks):
    disp = rec['canvas'] * 0.5 + 0.5
    ax.imshow(disp, cmap='gray', vmin=0, vmax=1)
    ys, xs = np.where(rec['gt512'] > 0.5)
    if len(xs):
        ax.scatter(xs, ys, s=1, c='lime', alpha=0.35)
    for rank, (b, sc, cv) in enumerate(zip(rec['boxes'][:3], rec['scores'][:3], rec['covs'][:3])):
        x0, y0, x1, y1 = b
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                               edgecolor=['red', 'orange', 'yellow'][rank], lw=2))
        ax.text(x0, max(0, y0 - 4), f'#{rank+1} q={sc:.2f} cov={cv:.2f}',
                color=['red', 'orange', 'yellow'][rank], fontsize=7)
    ax.set_title(rec['img_name'], fontsize=8); ax.axis('off')
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/prompt_suggestion_examples_{DS_NAME.lower()}.png', dpi=130, facecolor='white')
plt.show()